# TTC Subway Delay Data (2025): Cleaning + Outlier Removal (MAD)

This notebook implements a reproducible preprocessing pipeline for `ttc-subway-delay-data-2025.csv`:

1. Inspect raw categorical values (`Line`, `Bound`) before designing mappings.
2. Remove invalid rows first: any row where `Min Delay` or `Min Gap` is **missing**, **non-numeric**, or equals **0**.
3. Canonicalize:
   - `Line` → `Line_clean` (supports combinations like `YU/BD`)
   - `Bound` → `Bound_clean` (maps `N`, `NB`, `NORTH` → `N`, etc.)
4. Detect and drop outliers using a robust z-score based on **MAD (Median Absolute Deviation)**, computed per `Line_clean`.
5. Produce a statistical report comparing distributions before vs after outlier removal.
6. Save a cleaned CSV keeping original column order, plus:
   - `Bound_clean` as **second-to-last**
   - `Line_clean` as **last**


In [1]:
import numpy as np
import pandas as pd

## Configuration

In [2]:
FILE_PATH = "TTC Subway Delay Data since 2025.csv"
OUT_PATH = "TTC Subway Delay Data since 2025_cleaned.csv"
Z_THRESH = 6.0

# If True: raise if unexpected raw Bound formats appear after normalization
STRICT_RAW_BOUND_VALIDATION = False

## 1) Load + Inspect Raw Data

In [3]:
df_raw = pd.read_csv(FILE_PATH)
print("Original dataset shape:", df_raw.shape)

print("\n--- RAW CATEGORICAL INSPECTION ---")
print("\nRaw Line (top 20 counts incl NaN):")
print(df_raw["Line"].value_counts(dropna=False).head(20))

print("\nRaw Bound (top 20 counts incl NaN):")
print(df_raw["Bound"].value_counts(dropna=False).head(20))

# Validation of raw Bound formats (normalized)
raw_bound_norm = df_raw["Bound"].astype(str).str.strip().str.upper()
raw_bound_norm = raw_bound_norm.where(~df_raw["Bound"].isna(), other=np.nan)

allowed_raw_bound = {"N", "S", "E", "W", "NB", "SB", "EB", "WB", "NORTH", "SOUTH", "EAST", "WEST"}
unexpected_raw_bound = sorted(set(raw_bound_norm.dropna().unique()) - allowed_raw_bound)

if unexpected_raw_bound:
    msg = f"\nWARNING: Unexpected RAW Bound values found (normalized): {unexpected_raw_bound}"
    if STRICT_RAW_BOUND_VALIDATION:
        raise ValueError(msg)
    else:
        print(msg)

df = df_raw.copy()

Original dataset shape: (28191, 11)

--- RAW CATEGORICAL INSPECTION ---

Raw Line (top 20 counts incl NaN):
Line
YU              14358
BD              12216
SHP              1150
YU/BD             348
NaN                75
YUS/BD             16
YUS                 5
BD/YU               4
YU/ BD              3
BD/YUS              3
YU / BD             2
SRT                 2
YUS / BD            2
YU/BD LINES         2
YUS/ BD             1
YUS/ BD/ SHP        1
YU/BD/SHP           1
YU -BD LINES        1
29 DUFFERIN         1
Name: count, dtype: int64

Raw Bound (top 20 counts incl NaN):
Bound
NaN    10230
S       4831
E       4576
W       4277
N       4251
B         26
Name: count, dtype: int64



## 2) Remove zero / missing / non-numeric rows FIRST (`Min Delay`, `Min Gap`)

In [4]:
df["Min Delay"] = pd.to_numeric(df["Min Delay"], errors="coerce")
df["Min Gap"]   = pd.to_numeric(df["Min Gap"], errors="coerce")

before_n = len(df)
df = df.dropna(subset=["Min Delay", "Min Gap"])
df = df[(df["Min Delay"] > 0) & (df["Min Gap"] > 0)]
after_n = len(df)

print("\nAfter removing zero/missing/non-numeric (Min Delay/Min Gap):", df.shape, f"(dropped {before_n - after_n})")

# Validations
assert df["Min Delay"].notna().all() and df["Min Gap"].notna().all(), "Unexpected NaNs remain in Min Delay/Min Gap."
assert (df["Min Delay"] > 0).all() and (df["Min Gap"] > 0).all(), "Zeros/negatives remain in Min Delay/Min Gap." 


After removing zero/missing/non-numeric (Min Delay/Min Gap): (9530, 11) (dropped 18661)


## 3) Canonicalize `Line` and `Bound`

In [5]:
def canonical_line(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).upper().replace(" ", "")

    present = set()
    if "YU" in s:
        present.add("YU")
    if "BD" in s:
        present.add("BD")
    if "SHP" in s:
        present.add("SHP")
    if "SRT" in s:
        present.add("SRT")

    order = ["YU", "BD", "SHP", "SRT"]
    tokens = [t for t in order if t in present]

    return "/".join(tokens) if tokens else "UNKNOWN"


def canonical_bound(x):
    if pd.isna(x):
        return "UNKNOWN"

    s = str(x).strip().upper()

    mapping = {
        "N": "N", "NB": "N", "NORTH": "N",
        "S": "S", "SB": "S", "SOUTH": "S",
        "E": "E", "EB": "E", "EAST": "E",
        "W": "W", "WB": "W", "WEST": "W",
    }
    return mapping.get(s, "UNKNOWN")


df["Line_clean"] = df["Line"].apply(canonical_line)
df["Bound_clean"] = df["Bound"].apply(canonical_bound)

print("\n--- CANONICALIZED CATEGORICAL INSPECTION ---")
print("\nLine_clean counts:")
print(df["Line_clean"].value_counts(dropna=False))

print("\nBound_clean counts:")
print(df["Bound_clean"].value_counts(dropna=False))

# Validation: Bound_clean must be in allowed set
allowed_bound_clean = {"N", "S", "E", "W", "UNKNOWN"}
bad_bound_clean = sorted(set(df["Bound_clean"].dropna().unique()) - allowed_bound_clean)
assert not bad_bound_clean, f"Unexpected Bound_clean values: {bad_bound_clean}"

# Validation: Line_clean must be UNKNOWN or a valid slash-join of allowed tokens in canonical order
allowed_tokens = ["YU", "BD", "SHP", "SRT"]
allowed_line_clean = {"UNKNOWN"}
for r in range(1, 1 << len(allowed_tokens)):
    combo = [allowed_tokens[i] for i in range(len(allowed_tokens)) if (r >> i) & 1]
    allowed_line_clean.add("/".join(combo))

bad_line_clean = sorted(set(df["Line_clean"].dropna().unique()) - allowed_line_clean)
assert not bad_line_clean, f"Unexpected Line_clean values: {bad_line_clean}"

# Snapshot AFTER canonicalization, BEFORE outlier removal (for reporting)
df_before_outliers = df.copy()


--- CANONICALIZED CATEGORICAL INSPECTION ---

Line_clean counts:
Line_clean
YU         5217
BD         3946
SHP         366
UNKNOWN       1
Name: count, dtype: int64

Bound_clean counts:
Bound_clean
S          2808
N          2352
E          2220
W          2061
UNKNOWN      89
Name: count, dtype: int64


## 4) Outlier Detection with MAD (per `Line_clean`)

In [6]:
def mad_outlier_mask(series: pd.Series, z_thresh: float = Z_THRESH) -> pd.Series:
    x = series.to_numpy(dtype=float)
    med = np.median(x)
    mad = np.median(np.abs(x - med))

    if mad == 0 or np.isnan(mad):
        return pd.Series(False, index=series.index)

    robust_z = 0.6745 * (x - med) / mad
    return pd.Series(np.abs(robust_z) > z_thresh, index=series.index)


out_delay = df.groupby("Line_clean")["Min Delay"].transform(lambda s: mad_outlier_mask(s, z_thresh=Z_THRESH))
out_gap   = df.groupby("Line_clean")["Min Gap"].transform(lambda s: mad_outlier_mask(s, z_thresh=Z_THRESH))

mask_outlier = out_delay | out_gap
print("\nRows flagged as outliers:", int(mask_outlier.sum()))

df_clean = df.loc[~mask_outlier].copy()
print("Final cleaned dataset shape:", df_clean.shape)


Rows flagged as outliers: 376
Final cleaned dataset shape: (9154, 13)


## 5) Statistical Report: Distribution Changes

In [7]:
def distribution_report(df1: pd.DataFrame, df2: pd.DataFrame, col: str) -> None:
    print(f"\n=== Distribution Report: {col} ===")
    stats = pd.DataFrame({
        "Before": df1[col].describe(percentiles=[0.5, 0.9, 0.95, 0.99]),
        "After":  df2[col].describe(percentiles=[0.5, 0.9, 0.95, 0.99]),
    })
    pct = (stats["After"] - stats["Before"]) / stats["Before"].replace(0, np.nan) * 100
    stats["% Change"] = pct
    print(stats)

distribution_report(df_before_outliers, df_clean, "Min Delay")
distribution_report(df_before_outliers, df_clean, "Min Gap")

print("\n=== Mean Delay per Line (Before vs After) ===")
mean_compare = pd.DataFrame({
    "Before": df_before_outliers.groupby("Line_clean")["Min Delay"].mean(),
    "After":  df_clean.groupby("Line_clean")["Min Delay"].mean(),
})
mean_compare["% Change"] = (mean_compare["After"] - mean_compare["Before"]) / mean_compare["Before"].replace(0, np.nan) * 100
print(mean_compare.sort_index())


=== Distribution Report: Min Delay ===
            Before        After   % Change
count  9530.000000  9154.000000  -3.945435
mean      7.843547     5.853725 -25.368900
std      20.739775     3.716301 -82.081284
min       2.000000     2.000000   0.000000
50%       5.000000     5.000000   0.000000
90%      13.000000    11.000000 -15.384615
95%      19.000000    15.000000 -21.052632
99%      50.000000    20.000000 -60.000000
max     900.000000    22.000000 -97.555556

=== Distribution Report: Min Gap ===
            Before        After   % Change
count  9530.000000  9154.000000  -3.945435
mean     11.889717     9.871641 -16.973288
std      20.857685     4.077746 -80.449673
min       2.000000     2.000000   0.000000
50%       9.000000     9.000000   0.000000
90%      18.000000    16.000000 -11.111111
95%      24.000000    19.000000 -20.833333
99%      54.000000    24.000000 -55.555556
max     906.000000    27.000000 -97.019868

=== Mean Delay per Line (Before vs After) ===
              B

## 6) Column Order + Save

Keep original column order and append `Bound_clean` (second-to-last) and `Line_clean` (last).


In [8]:
original_cols = list(df_raw.columns)
df_clean = df_clean[original_cols + ["Bound_clean", "Line_clean"]]

assert df_clean.columns[-2] == "Bound_clean"
assert df_clean.columns[-1] == "Line_clean"

df_clean.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Saved: TTC Subway Delay Data since 2025_cleaned.csv
